In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import unicodedata
import pyarrow as pa
import pyarrow.parquet as pq
import re
import os
from difflib import get_close_matches

In [2]:
# Importar RIPS
os.listdir('/kaggle/input/datasets/geraldinelaverde/rips-inicial')
ruta_archivo = '/kaggle/input/datasets/geraldinelaverde/rips-inicial/df_rips_2.parquet'
df_rips = pd.read_parquet(ruta_archivo)
df_rips.head()

,Departamento,Municipio,Año,TipoAtencion,Diagnostico,NumeroAtenciones,CÓDIGO DEPARTAMENTO,CÓDIGO MUNICIPIO,Cod_Diagnostico,Diagnostico_Nombre
0,05 - Antioquia,05212 - Copacabana,2012,CONSULTAS,F431 - TRASTORNO DE ESTRÃ‰S POSTRAUMATICO,2,05,05212,F431,TRASTORNO DE ESTRÃ‰S POSTRAUMATICO
1,05 - Antioquia,05576 - Pueblorrico,2011,CONSULTAS,"C779 - TUMOR MALIGNO DEL GANGLIO LINFATICO, SI...",1,05,05576,C779,"TUMOR MALIGNO DEL GANGLIO LINFATICO, SITIO NO ..."
2,05 - Antioquia,05086 - Belmira,2014,PROCEDIMIENTOS DE SALUD,Z340 - SUPERVISION DE PRIMER EMBARAZO NORMAL,48,05,05086,Z340,SUPERVISION DE PRIMER EMBARAZO NORMAL
3,05 - Antioquia,05490 - Necoclí,2015,CONSULTAS,M545 - LUMBAGO NO ESPECIFICADO,961,05,05490,M545,LUMBAGO NO ESPECIFICADO
4,05 - Antioquia,05212 - Copacabana,2012,CONSULTAS,F510 - INSOMNIO NO ORGANICO,32,05,05212,F510,INSOMNIO NO ORGANICO


In [3]:
#Importar divipola
os.listdir('/kaggle/input/datasets/nicolasacostaa/municipios-divipola')
ruta_davipola = '/kaggle/input/datasets/nicolasacostaa/municipios-divipola/df_Davipola.parquet'
df_Davipola = pd.read_parquet(ruta_davipola)
df_Davipola.head()

,CÓDIGO DEPARTAMENTO,NOMBRE DEPARTAMENTO,CÓDIGO MUNICIPIO,NOMBRE MUNICIPIO
0,91,AMAZONAS,91263,EL ENCANTO
1,91,AMAZONAS,91405,LA CHORRERA
2,91,AMAZONAS,91407,LA PEDRERA
3,91,AMAZONAS,91430,LA VICTORIA
4,91,AMAZONAS,91001,LETICIA


In [4]:
# -------------------------------------------------
# 2) Unir con DIVIPOLA (la parte clave)
# -------------------------------------------------

df_rips = df_rips.merge(
    df_Davipola,
    on=['CÓDIGO DEPARTAMENTO','CÓDIGO MUNICIPIO'],
    how='left'
)

# Guardar dataframe como parquet
df_rips.to_parquet('/kaggle/working/df_rips_3.parquet', index=False)

In [5]:
# -------------------------------------------------
# 3) Crear ANO y MES desde Año (para tu modelo)
# -------------------------------------------------

df_rips['ANO'] = df_rips['Año'].astype(int)
df_rips['MES'] = 1  # RIPS no trae mes, se deja fijo si luego lo cruzas con giros
# Guardar dataframe como parquet
df_rips.to_parquet('/kaggle/working/df_rips_4.parquet', index=False)

In [6]:
# -------------------------------------------------
# 4) Dejar dataset final 
# -------------------------------------------------

df_rips_final = df_rips[['ANO','MES','CÓDIGO DEPARTAMENTO','NOMBRE DEPARTAMENTO',
                         'CÓDIGO MUNICIPIO','NOMBRE MUNICIPIO','TipoAtencion',
                         'Cod_Diagnostico','Diagnostico_Nombre','NumeroAtenciones']].reset_index(drop=True)

print("✅ Dataset geográfico listo")

df_rips_final.head()

df_rips_final.info()
# Guardar dataframe como parquet
df_rips_final.to_parquet('/kaggle/working/df_rips_final.parquet', index=False)

✅ Dataset geográfico listo
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 38000000 entries, 0 to 37999999
Data columns (total 10 columns):
 #   Column               Dtype 
---  ------               ----- 
 0   ANO                  int64 
 1   MES                  int64 
 2   CÓDIGO DEPARTAMENTO  object
 3   NOMBRE DEPARTAMENTO  object
 4   CÓDIGO MUNICIPIO     object
 5   NOMBRE MUNICIPIO     object
 6   TipoAtencion         object
 7   Cod_Diagnostico      object
 8   Diagnostico_Nombre   object
 9   NumeroAtenciones     Int64 
dtypes: Int64(1), int64(2), object(7)
memory usage: 2.9+ GB
